# MedGAN Synthetic Data Generation for MIMIC-III

_Last updated: 2026-03-08_

This notebook trains MedGAN on your MIMIC-III data and generates synthetic patients.

## What You'll Need

1. **MIMIC-III Access**: Download these files from PhysioNet:
   - `PATIENTS.csv` — patient demographics
   - `ADMISSIONS.csv` — hospital admission records
   - `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

2. **Google Colab**: Free tier works. GPU recommended but not required (MedGAN is lightweight).

## How It Works

MedGAN uses a two-phase training process:
1. **Phase 1 — Autoencoder pretraining**: A linear autoencoder learns a compressed representation of binary diagnosis vectors.
2. **Phase 2 — Adversarial training**: A generator maps random noise to the autoencoder's latent space, and the decoder projects back to binary codes. A discriminator (with minibatch averaging) distinguishes real vs. synthetic records.

Each patient is represented as a flat **bag-of-codes** (binary vector indicating which ICD-9 codes appear across all visits). MedGAN does not model visit structure.

## References

- [MedGAN Paper](https://arxiv.org/abs/1703.06490) — Choi et al., MLHC 2017
- [PyHealth Documentation](https://pyhealth.readthedocs.io/)
- [MIMIC-III Access](https://physionet.org/content/mimiciii/)

---
# 1. Setup & Installation

In [ ]:
import subprocess
import sys
import os

# Record Colab's pre-installed numpy version so we can detect if it changed.
_np_before = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True,
).stdout.strip()

FORK = 'jalengg'
BRANCH = 'medgan-pr-integration'
install_url = f"git+https://github.com/{FORK}/PyHealth.git@{BRANCH}"

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "pyhealth", "-y"],
    capture_output=True, text=True,
)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", install_url, "--quiet", "--no-cache-dir"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("PyHealth installation failed.")

# Check if numpy was upgraded. If so, the already-loaded C extensions are
# stale and will cause "cannot import name '_center'" errors. The only
# reliable fix is to restart the kernel so Python loads the new .so files.
_np_after = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True,
).stdout.strip()

if _np_after != _np_before:
    print(f"numpy upgraded {_np_before} -> {_np_after}. Restarting kernel...")
    print(">>> After restart, re-run this cell. It will skip the restart on the second run. <<<")
    os.kill(os.getpid(), 9)  # force kernel restart

print(f"PyHealth installed from {FORK}/{BRANCH}")
print(f"numpy: {_np_after}")

In [ ]:
import os
import shutil
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive, files

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. MedGAN is lightweight and runs fine on CPU.")

In [ ]:
# Mount Google Drive for persistent storage
print("Mounting Google Drive...")
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("Drive already mounted")
print("Google Drive mounted")

BASE_DIR       = '/content/drive/MyDrive/MedGAN_Training'
DATA_DIR       = f'{BASE_DIR}/data'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
OUTPUT_DIR     = f'{BASE_DIR}/output'

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"\nDirectory structure:")
print(f"  Base: {BASE_DIR}")
print(f"  Data: {DATA_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

---
# 2. Configuration

Configure your training and generation parameters below.

**For Quick Demo (recommended first time):**
- Leave defaults (10 AE epochs, 20 GAN epochs, 50 samples)

**For Production Quality:**
- Set `AE_EPOCHS = 100`, `GAN_EPOCHS = 200`
- Set `N_SYNTHETIC_SAMPLES = 10000`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Training
AE_EPOCHS  = 10      # Demo: 10, Production: 100
GAN_EPOCHS = 20      # Demo: 20, Production: 200
BATCH_SIZE = 128     # Larger batches stabilize GAN training
AE_LR      = 0.001   # Autoencoder learning rate
GAN_LR     = 0.001   # GAN learning rate

# Model architecture
LATENT_DIM        = 128  # Generator noise / AE latent dimension
HIDDEN_DIM        = 128  # Generator hidden width
AE_HIDDEN_DIM     = 128  # Autoencoder hidden width
DISC_HIDDEN_DIM   = 256  # Discriminator hidden width

# Generation
N_SYNTHETIC_SAMPLES = 50  # Demo: 50, Production: 10000

# Display
print("=" * 60)
print("MEDGAN CONFIGURATION")
print("=" * 60)
print(f"Training:")
print(f"  AE epochs: {AE_EPOCHS}  |  GAN epochs: {GAN_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  AE LR: {AE_LR}  |  GAN LR: {GAN_LR}")
print(f"\nGeneration:")
print(f"  Synthetic samples: {N_SYNTHETIC_SAMPLES}")
print("=" * 60)

---
# 3. Data Upload

Upload your MIMIC-III CSV files:

1. `PATIENTS.csv` — patient demographics
2. `ADMISSIONS.csv` — admission records
3. `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

Files persist across Colab sessions when saved to Google Drive.

In [ ]:
required_files = {
    'PATIENTS.csv':      'Patient demographics',
    'ADMISSIONS.csv':    'Admission records',
    'DIAGNOSES_ICD.csv': 'ICD-9 diagnosis codes',
}
existing = {f: os.path.exists(f'{DATA_DIR}/{f}') for f in required_files}
missing  = [f for f, ok in existing.items() if not ok]

if not missing:
    print("All MIMIC-III files found in Drive (no upload needed):")
    for fname in required_files:
        size_mb = os.path.getsize(f'{DATA_DIR}/{fname}') / 1024 / 1024
        print(f"    {fname}  ({size_mb:.1f} MB)")
    print(f"\nFiles reused from: {DATA_DIR}")
else:
    print("MIMIC-III file status:")
    for fname, desc in required_files.items():
        mark = "OK" if existing[fname] else "MISSING"
        print(f"  [{mark}]  {fname} - {desc}")

    print(f"\nUploading {len(missing)} missing file(s)...")
    uploaded = files.upload()

    for uploaded_name, data in uploaded.items():
        matched = None
        for req in required_files:
            base = req.replace('.csv', '')
            if base in uploaded_name and uploaded_name.endswith('.csv'):
                matched = req
                break
        if matched:
            tmp = f'/content/{uploaded_name}'
            with open(tmp, 'wb') as f:
                f.write(data)
            dest = f'{DATA_DIR}/{matched}'
            shutil.copy(tmp, dest)
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"  Saved {matched} ({size_mb:.1f} MB)")
        else:
            print(f"  Unrecognised file: {uploaded_name} (skipped)")

    missing = [f for f in required_files if not os.path.exists(f'{DATA_DIR}/{f}')]
    if missing:
        raise FileNotFoundError(
            f"Still missing: {missing}. Please re-run this cell."
        )
    print("\nAll 3 MIMIC-III files present.")

In [ ]:
print("Validating MIMIC-III files...")

_patients = pd.read_csv(f'{DATA_DIR}/PATIENTS.csv', nrows=5)
assert 'SUBJECT_ID' in _patients.columns
print(f"  PATIENTS.csv: {len(_patients.columns)} columns")

_admissions = pd.read_csv(f'{DATA_DIR}/ADMISSIONS.csv', nrows=5)
assert 'HADM_ID' in _admissions.columns
print(f"  ADMISSIONS.csv: {len(_admissions.columns)} columns")

_diagnoses = pd.read_csv(f'{DATA_DIR}/DIAGNOSES_ICD.csv', nrows=5)
assert 'ICD9_CODE' in _diagnoses.columns
print(f"  DIAGNOSES_ICD.csv: {len(_diagnoses.columns)} columns")

del _patients, _admissions, _diagnoses
print("\nAll files validated.")

---
# 4. Training

**What happens during training:**

1. **Dataset loading**: PyHealth reads MIMIC-III and creates one sample per patient (flat list of all ICD-9 codes across all visits).
2. **Multi-hot encoding**: The `MultiHotProcessor` converts each patient's code list into a binary vector of shape `(vocab_size,)`.
3. **Phase 1 — AE pretraining**: A linear autoencoder learns to reconstruct the binary vectors using BCE loss.
4. **Phase 2 — GAN training**: The generator maps noise to the AE latent space. The AE decoder projects to binary codes. The discriminator (with minibatch averaging) provides the adversarial signal via standard BCE loss.
5. **Checkpoint**: Best and final checkpoints saved to `CHECKPOINT_DIR`.

In [ ]:
from pyhealth.datasets import MIMIC3Dataset, split_by_patient
from pyhealth.tasks import medgan_generation_mimic3_fn
from pyhealth.models.generators.medgan import MedGAN

print("Loading MIMIC-III dataset...")
base_dataset = MIMIC3Dataset(
    root=DATA_DIR,
    tables=["diagnoses_icd"],
    dev=False,
)
print(f"Loaded {len(base_dataset.unique_patient_ids)} patients")

print("Applying MedGAN generation task...")
sample_dataset = base_dataset.set_task(medgan_generation_mimic3_fn)
print(f"Eligible patients: {len(sample_dataset)}")

train_dataset, val_dataset, _ = split_by_patient(sample_dataset, [0.8, 0.1, 0.1])
print(f"Split: {len(train_dataset)} train / {len(val_dataset)} val")

In [ ]:
# Initialize and train MedGAN
print("Initializing MedGAN model...")
model = MedGAN(
    dataset=sample_dataset,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
    autoencoder_hidden_dim=AE_HIDDEN_DIM,
    discriminator_hidden_dim=DISC_HIDDEN_DIM,
    batch_size=BATCH_SIZE,
    ae_epochs=AE_EPOCHS,
    gan_epochs=GAN_EPOCHS,
    ae_lr=AE_LR,
    gan_lr=GAN_LR,
    save_dir=CHECKPOINT_DIR,
)
print(f"Vocabulary size: {model.input_dim} ICD-9 codes")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

print("\nStarting training...")
print("=" * 60)
model.train_model(train_dataset, val_dataset)
print("=" * 60)
print(f"Training complete! Checkpoints saved to: {CHECKPOINT_DIR}")

---
# 5. Generation

**How generation works:**

1. Sample random noise vectors from a standard normal distribution
2. Generator maps noise to the autoencoder's latent space
3. AE decoder projects latent vectors to binary code space (sigmoid output)
4. Threshold at 0.5 to produce binary vectors
5. Map active indices back to ICD-9 code strings

Each synthetic patient is a flat **bag-of-codes** (no visit structure).

In [ ]:
print(f"Generating {N_SYNTHETIC_SAMPLES} synthetic patients...")
synthetic = model.synthesize_dataset(num_samples=N_SYNTHETIC_SAMPLES)
print(f"Generated {len(synthetic)} synthetic patients")

# Summary statistics
codes_per_patient = [len(p['visits']) for p in synthetic]
avg_codes = np.mean(codes_per_patient)
non_empty = sum(1 for c in codes_per_patient if c > 0)

print(f"\nStatistics:")
print(f"  Non-empty patients: {non_empty}/{len(synthetic)}")
print(f"  Avg codes per patient: {avg_codes:.2f}")
print(f"  Min codes: {min(codes_per_patient)}")
print(f"  Max codes: {max(codes_per_patient)}")

# Preview
preview = []
for p in synthetic[:10]:
    sample_codes = ', '.join(p['visits'][:5]) + ('...' if len(p['visits']) > 5 else '')
    preview.append({
        'patient_id': p['patient_id'],
        'n_codes': len(p['visits']),
        'sample_codes': sample_codes or '(empty)',
    })
display(pd.DataFrame(preview))

In [ ]:
# Save as CSV (flat SUBJECT_ID, ICD9_CODE — one row per code per patient)
rows = []
for p in synthetic:
    for code in p['visits']:
        rows.append({'SUBJECT_ID': p['patient_id'], 'ICD9_CODE': code})

df_synthetic = pd.DataFrame(rows)
csv_path = f'{OUTPUT_DIR}/medgan_synthetic_data.csv'
df_synthetic.to_csv(csv_path, index=False)

print(f"{len(df_synthetic):,} records saved to: {csv_path}")
print(f"Columns: SUBJECT_ID, ICD9_CODE")
print(f"\nSample rows:")
display(df_synthetic.head(8))

---
# 6. Results & Download

In [ ]:
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

unique_patients = df_synthetic['SUBJECT_ID'].nunique()
print(f"\nUnique patients: {unique_patients} out of {N_SYNTHETIC_SAMPLES} requested")

# Check for empty values
empty_subjects = df_synthetic['SUBJECT_ID'].isna().sum()
empty_codes = df_synthetic['ICD9_CODE'].isna().sum()
print(f"\nEmpty values:")
print(f"  Subject IDs: {empty_subjects} (should be 0)")
print(f"  ICD9 codes: {empty_codes} (should be 0)")

# Code distribution
codes_per_patient = df_synthetic.groupby('SUBJECT_ID').size()
print(f"\nCodes per patient:")
print(f"  Min: {codes_per_patient.min()}")
print(f"  Max: {codes_per_patient.max()}")
print(f"  Mean: {codes_per_patient.mean():.2f}")
print(f"  Median: {codes_per_patient.median():.2f}")

# Unique codes used
unique_codes = df_synthetic['ICD9_CODE'].nunique()
print(f"\nUnique ICD-9 codes in synthetic data: {unique_codes}")

# Heterogeneity
unique_profiles = len(set(
    tuple(sorted(df_synthetic[df_synthetic['SUBJECT_ID'] == pid]['ICD9_CODE'].tolist()))
    for pid in df_synthetic['SUBJECT_ID'].unique()
))
print(f"Unique patient profiles: {unique_profiles}/{unique_patients} "
      f"({unique_profiles/max(unique_patients,1)*100:.1f}%)")

print("\n" + "=" * 60)
print("QUALITY CHECKS COMPLETE")
print("=" * 60)

In [ ]:
print("=" * 60)
print("DOWNLOAD SYNTHETIC DATA")
print("=" * 60)

print(f"\nYour synthetic data is ready:")
print(f"  File: medgan_synthetic_data.csv")
print(f"  Patients: {unique_patients:,}")
print(f"  Total records: {len(df_synthetic):,}")
print(f"  Size: {os.path.getsize(csv_path) / (1024*1024):.2f} MB")

print(f"\nDownloading...")
files.download(csv_path)

print(f"\nFile also saved in Google Drive:")
print(f"  {csv_path}")

---
## Congratulations!

You've successfully:
1. Trained a MedGAN model on your MIMIC-III data
2. Generated synthetic patients
3. Validated the synthetic data quality
4. Downloaded the CSV file

## Next Steps

**Use your synthetic data:**
- Train predictive models (readmission, mortality, etc.)
- Evaluate data utility via Train-on-Synthetic, Test-on-Real (TSTR)
- Share data without privacy concerns

**Generate more samples:**
- Change `N_SYNTHETIC_SAMPLES` and re-run Section 5
- No need to retrain if the model is still in memory

**Production training:**
- Set `AE_EPOCHS = 100`, `GAN_EPOCHS = 200`
- Set `N_SYNTHETIC_SAMPLES = 10000`

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| All synthetic patients empty | Increase `GAN_EPOCHS` |
| All patients have identical codes | Increase `GAN_EPOCHS`, check `AE_EPOCHS` reduced loss |
| Training loss not decreasing | Try `GAN_LR = 0.0002` |
| Out of memory | Reduce `BATCH_SIZE` to 64 or 32 |

## References

- [MedGAN Paper](https://arxiv.org/abs/1703.06490) — Choi et al., MLHC 2017
- [PyHealth Docs](https://pyhealth.readthedocs.io/)
- [GitHub Issues](https://github.com/sunlabuiuc/PyHealth/issues)